# Notebook 00 — Setup y verificación del entorno

**Notebook:** `00_setup.ipynb`  
**TFM:** Bioinformática estructural — Familia RAS  
**Objetivo:** Verificar que el entorno de trabajo está correctamente instalado y que todos los componentes necesarios para ejecutar el pipeline del TFM están disponibles.

Este notebook es un **diagnóstico** que debes ejecutar antes de empezar con los **notebooks de análisis** (01, 02, 03, 04).

---

## ¿Qué verifica este notebook?

| Sección | Qué comprueba |
|---------|---------------|
| 1. Python | Versión >= 3.9 |
| 2. Librerías | Todas las dependencias del pipeline |
| 3. Paquete tfm_ras | Submódulos: cosmic_loader, alignment, structural, visualization |
| 4. Configuración | `configs/config.yaml` legible y válido |
| 5. Datos | Ficheros de entrada presentes (secuencias, PDBs, dataset sintético) |
| 6. Herramientas externas | clustalo/mafft/muscle, mkdssp (con degradación graceful) |
| 7. Directorios de salida | `data/processed/` y `figures/` creados |
| 8. Resumen | Estado global del entorno |


---
## Sección 1: Versión de Python

El pipeline requiere **Python 3.9 o superior**. Versiones anteriores no son compatibles con algunas funcionalidades de `pathlib`, f-strings avanzadas y anotaciones de tipos usadas en el código.

In [91]:
# ============================================================
# CHUNK 1.1: Verificar versión de Python
# ============================================================

import sys  # módulo para acceder a información del intérprete

print("=" * 55)
print("VERIFICACIÓN DEL ENTORNO — TFM RAS")
print("=" * 55)
print()

# sys.version es una cadena de texto con la versión completa de Python.
# sys.version_info es una namedtuple con major, minor, micro separados.
print(f"Python: {sys.version}")

if sys.version_info >= (3, 9):
    print("  [OK] Python >= 3.9")
else:
    # Si la versión es antigua, mostramos un error claro en lugar de
    # dejar que el error aparezca más tarde de forma críptica.
    print("  [ERROR] Python < 3.9. Actualiza el entorno conda.")
    print("  Ejecuta: conda env create -f environment.yml")

VERIFICACIÓN DEL ENTORNO — TFM RAS

Python: 3.11.15 | packaged by conda-forge | (main, Mar  5 2026, 16:58:53) [Clang 19.1.7 ]
  [OK] Python >= 3.9


---
## Sección 2: Librerías del pipeline

El pipeline usa las siguientes librerías científicas:

| Librería | Uso principal |
|----------|---------------|
| `pandas` | Manipulación de tablas (DataFrames) |
| `numpy` | Álgebra lineal, arrays numéricos |
| `Bio` | Parseo de PDB, MSA, secuencias FASTA |
| `matplotlib` | Figuras 2D (base) |
| `seaborn` | Heatmaps, clustermaps, scatter plots estadísticos |
| `scipy` | Cálculos estadísticos y de distancias |
| `py3Dmol` | Visualizaciones 3D interactivas de proteínas (HTML) |
| `yaml` (PyYAML) | Lectura del fichero de configuración |
| `requests` | Descarga de ficheros (UniProt, PDB) |
| `tqdm` | Barras de progreso en bucles |

Si alguna falta, instálala con: `pip install <nombre_librería>`

In [92]:
# ============================================================
# CHUNK 2.1: Verificar disponibilidad de librerías
# ============================================================

import sys
import pandas as pd
import numpy as np
import Bio
import matplotlib
import seaborn
import scipy
import py3Dmol
import yaml
import requests
import tqdm

# Definimos una lista de tuplas (nombre_import, nombre_pip).
# El nombre_import es lo que se usa en 'import ...'; a veces difiere
# del nombre del paquete pip (p. ej. 'Bio' -> 'biopython').
LIBS = [
    ("pandas",      "pandas"),
    ("numpy",       "numpy"),
    ("Bio",         "biopython"),
    ("matplotlib",  "matplotlib"),
    ("seaborn",     "seaborn"),
    ("scipy",       "scipy"),
    ("py3Dmol",     "py3Dmol"),
    ("yaml",        "PyYAML"),
    ("requests",    "requests"),
    ("tqdm",        "tqdm"),
]

print("Verificando librerías...")
print("-" * 40)

all_libs_ok = True  # bandera para saber si todas las librerías están presentes

for import_name, pip_name in LIBS:
    try:
        # __import__(nombre) importa un módulo dinámicamente dado su nombre como cadena.
        # Esto es equivalente a escribir 'import pandas' pero sin conocer el nombre
        # en tiempo de escritura del código.
        mod = __import__(import_name) 

        # getattr(objeto, "__version__", "?") devuelve el atributo __version__
        # del módulo si existe, o '?' si el módulo no tiene versión definida.
        version = getattr(mod, "__version__", "?")

        print(f"  [OK] {pip_name:15s} v{version}")
    except ImportError:
        # ImportError se lanza cuando el módulo no está instalado.
        print(f"  [FALTA] {pip_name:15s} → pip install {pip_name}")
        all_libs_ok = False  # marcamos que falta al menos una librería

print("-" * 40)
if all_libs_ok:
    print("Todas las librerías están disponibles.")
else:
    print("ADVERTENCIA: faltan librerías. Instala con 'pip install <nombre>'.")

Verificando librerías...
----------------------------------------
  [OK] pandas          v3.0.3
  [OK] numpy           v2.4.6
  [OK] biopython       v1.87
  [OK] matplotlib      v3.10.9
  [OK] seaborn         v0.13.2
  [OK] scipy           v1.17.1
  [OK] py3Dmol         v2.5.4
  [OK] PyYAML          v6.0.3
  [OK] requests        v2.34.2
  [OK] tqdm            v4.67.3
----------------------------------------
Todas las librerías están disponibles.


---
## Sección 3: Paquete `tfm_ras`

El paquete `tfm_ras` contiene todo el código del pipeline organizado en módulos:

| Módulo | Descripción |
|--------|-------------|
| `cosmic_loader` | Carga, filtrado y curación de datos de COSMIC |
| `alignment` | MSA (alineamiento múltiple de secuencias) y mapa posicional |
| `structural` | Cálculo de SASA, distancia al sitio activo, entropía |
| `visualization` | Figuras 2D (heatmap, scatter) y 3D interactivas (py3Dmol) |
| `config` | Carga del fichero de configuración YAML |

Si obtienes `ModuleNotFoundError`, instala el paquete en modo desarrollo: `pip install -e .`

In [93]:
# ============================================================
# CHUNK 3.1: Verificar importación del paquete tfm_ras
# ============================================================

print("Verificando paquete tfm_ras...")
print("-" * 40)

# Importamos los submódulos principales del paquete.

modules_ok = True 

try:
    # Importamos el paquete principal para verificar que está instalado.
    import tfm_ras
    print(f"  [OK] tfm_ras v{tfm_ras.__version__}")
except ImportError as e:
    print(f"  [ERROR] tfm_ras no encontrado: {e}")
    print("          → Instala con: pip install -e .")
    modules_ok = False

# Lista de submódulos a verificar
SUBMODULES = [
    ("tfm_ras.config",         "carga de configuración YAML"),
    ("tfm_ras.cosmic_loader",  "curación de datos COSMIC"),
    ("tfm_ras.alignment",      "MSA y mapa posicional"),
    ("tfm_ras.structural",     "SASA, distancia, entropía"),
    ("tfm_ras.visualization",  "figuras 2D y 3D"),
]

for module_name, description in SUBMODULES:
    try:
        # __import__(nombre) importa el módulo completo incluyendo el paquete padre.
        # Para submódulos con puntos (e.g. 'tfm_ras.config') necesitamos
        # fromlist=[''] para que devuelva el submódulo en lugar del paquete raíz.
        mod = __import__(module_name, fromlist=[""])  # importación del submódulo
        print(f"  [OK] {module_name:35s} — {description}")
    except ImportError as e:
        print(f"  [ERROR] {module_name}: {e}")
        modules_ok = False

print("-" * 40)
if modules_ok:
    print("Paquete tfm_ras importado correctamente.")
else:
    print("ERROR: algunos módulos no se pudieron importar.")
    print("Revisa el código fuente en src/tfm_ras/")

Verificando paquete tfm_ras...
----------------------------------------
  [OK] tfm_ras v0.1.0
  [OK] tfm_ras.config                      — carga de configuración YAML
  [OK] tfm_ras.cosmic_loader               — curación de datos COSMIC
  [OK] tfm_ras.alignment                   — MSA y mapa posicional
  [OK] tfm_ras.structural                  — SASA, distancia, entropía
  [OK] tfm_ras.visualization               — figuras 2D y 3D
----------------------------------------
Paquete tfm_ras importado correctamente.


---
## Sección 4: Configuración del proyecto

El fichero `configs/config.yaml` centraliza todos los parámetros del pipeline:
- **Genes**: KRAS, HRAS, NRAS con sus IDs de UniProt y PDB
- **Hotspots**: posiciones G12, G13, Q61, A146, K117, A59
- **Umbrales**: criterios de recurrencia para la detección de hotspots
- **Regiones funcionales**: P-loop, Switch I, Switch II, NKxD y SAK

La función `load_config()` del módulo `config.py` carga este YAML y lo devuelve como diccionario de Python.

In [94]:
# ============================================================
# CHUNK 4.1: Cargar y verificar la configuración del proyecto
# ============================================================

from pathlib import Path  # para construir rutas independientes del sistema operativo
from tfm_ras.config import project_root, load_config  # funciones del módulo de configuración

# project_root() resuelve la ruta raíz del proyecto de forma dinámica.
# Internamente usa Path(__file__).resolve().parent.parent.parent para
# subir tres niveles desde src/tfm_ras/config.py hasta plantilla_v2/.
ROOT = project_root()  # ruta raíz del proyecto

print(f"Raíz del proyecto: {ROOT}")
print()

# Intentamos cargar el fichero de configuración.
# load_config() lanza FileNotFoundError si el YAML no existe.
try:
    CFG = load_config(ROOT / 'configs' / 'config.yaml')

    # Verificamos que tiene las secciones principales.
    # .get(clave) devuelve None si la clave no existe (sin lanzar excepción).
    expected_sections = ["family", "hotspots", "thresholds", "active_site"]
    print("Configuración cargada. Secciones encontradas:")
    for section in expected_sections:
        value = CFG.get(section)  # intenta obtener la sección del diccionario
        if value is not None:
            print(f"  [OK] {section}")
        else:
            print(f"  [FALTA] {section} — sección no encontrada en config.yaml")

    # Mostramos los genes configurados.
    # CFG["family"]["members"] es un diccionario con claves KRAS, HRAS, NRAS.
    genes = list(CFG.get("family", {}).get("members", {}).keys())
    print(f"\n  Genes configurados: {genes}")

    # Mostramos los hotspots configurados.
    hotspots = CFG.get("hotspots", [])
    hotspot_str = ", ".join([f'{h["residue"]}{h["position"]}' for h in hotspots])
    print(f"  Hotspots: {hotspot_str}")

    print("\n  [OK] Configuración válida.")

except FileNotFoundError as e:
    # FileNotFoundError: el fichero YAML no existe en la ruta esperada.
    print(f"  [ERROR] No se encontró config.yaml: {e}")
    print("  Verifica que ejecutas el notebook desde el directorio correcto")

Raíz del proyecto: /Users/rachi/Desktop/TFM/tfm_ras_mutations

Configuración cargada. Secciones encontradas:
  [OK] family
  [OK] hotspots
  [OK] thresholds
  [OK] active_site

  Genes configurados: ['KRAS', 'HRAS', 'NRAS']
  Hotspots: G12, G13, Q61, A146, K117, A59

  [OK] Configuración válida.


---
## Sección 5: Ficheros de datos de entrada

El pipeline necesita tres tipos de ficheros de datos de entrada:

1. **Dataset COSMIC** (`data/raw/cosmic/`): tabla de mutaciones de COSMIC. Es la base de datos central del análisis. 

2. **Secuencias FASTA** (`data/external/sequences/`): secuencias canónicas en UniProt para KRAS (P01116), HRAS (P01112) y NRAS (P01111). Son las entradas del alineamiento múltiple (MSA).

3. **Estructuras PDB** (`data/external/pdb/`): ficheros de coordenadas atómicas de las proteínas. Se usan para calcular SASA y distancias al sitio activo.

In [95]:
# ============================================================
# CHUNK 5.1: Descarga de estructuras PDB
# ============================================================

from Bio.PDB import PDBList, PDBParser
from tfm_ras import cosmic_loader

CFG = load_config(ROOT / 'configs' / 'config.yaml')
EXTERNAL = ROOT / CFG['paths']['data_external'] / 'pdb'

pdbl = PDBList()
parser = PDBParser(QUIET=True)

ras_genes = cosmic_loader.RAS_GENES

for gene in ras_genes:

    gene_cfg = CFG['family']['members'][gene]

    pdb_ids = (
        gene_cfg['pdbs']['GTP_structure']['id'],
        gene_cfg['pdbs']['GDP_structure']['id']
    )

    for pdb_id in pdb_ids:

        pdb_file = pdbl.retrieve_pdb_file(
            pdb_id,
            pdir=EXTERNAL,
            file_format='pdb'
        )

        structure = parser.get_structure(
            pdb_id,
            pdb_file
        )

        print(f'{gene} - {pdb_id}: parse OK')

Structure exists: '/Users/rachi/Desktop/TFM/tfm_ras_mutations/data/external/pdb/pdb5uk9.ent' 
KRAS - 5UK9: parse OK
Structure exists: '/Users/rachi/Desktop/TFM/tfm_ras_mutations/data/external/pdb/pdb4obe.ent' 
KRAS - 4OBE: parse OK
Structure exists: '/Users/rachi/Desktop/TFM/tfm_ras_mutations/data/external/pdb/pdb3k8y.ent' 
HRAS - 3K8Y: parse OK
Structure exists: '/Users/rachi/Desktop/TFM/tfm_ras_mutations/data/external/pdb/pdb4q21.ent' 
HRAS - 4Q21: parse OK
Structure exists: '/Users/rachi/Desktop/TFM/tfm_ras_mutations/data/external/pdb/pdb5uhv.ent' 
NRAS - 5UHV: parse OK
Structure exists: '/Users/rachi/Desktop/TFM/tfm_ras_mutations/data/external/pdb/pdb3con.ent' 
NRAS - 3CON: parse OK


In [96]:
# ============================================================
# CHUNK 5.2: Descarga de secuencias de UniProt
# ============================================================

from Bio.PDB import PDBList, PDBParser
from tfm_ras import cosmic_loader
from tfm_ras.alignment import fetch_uniprot_sequence

# Extraemos los identificadores UniProt de cada isoforma RAS desde la configuracion.
uniprot_ids = {
    gene: info["uniprot_id"]
    for gene, info in CFG["family"]["members"].items()
}

sequences = {
    gene: fetch_uniprot_sequence(uniprot_id)
    for gene, uniprot_id in uniprot_ids.items()
}

print("[OK] Secuencias cargadas")

[OK] Secuencias cargadas


In [97]:
# ============================================================
# CHUNK 5.3: Verificar ficheros de datos de entrada
# ============================================================

RAW = ROOT / CFG['paths']['data_raw']
EXTERNAL = ROOT / "data" / "external"

COSMIC = RAW / "cosmic"
SEQUENCES = EXTERNAL / "sequences"
PDB = EXTERNAL / "pdb"


raw_filename = (COSMIC
                   / f"Cosmic_MutantCensus_Tsv_{CFG['cosmic']['version']}_{CFG['cosmic']['human_reference_genome']}" 
                   / CFG['cosmic']['raw_filename'].format(**CFG['cosmic']))

classification_filename = (COSMIC
                             / f"Cosmic_Classification_Tsv_{CFG['cosmic']['version']}_{CFG['cosmic']['human_reference_genome']}" 
                             / CFG['cosmic']['classification_filename'].format(**CFG['cosmic']))


print("Verificando ficheros de datos de entrada...")
print("-" * 60)

# Definimos los ficheros requeridos con una descripción para el error.

REQUIRED_FILES = [
    # Dataset COSMIC
    (raw_filename,
     "Dataset de mutaciones COSMIC"),
     (classification_filename,
     "Dataset de clasificación COSMIC"),

    # Secuencias canónicas de UniProt (FASTA)
    (SEQUENCES / "P01116.fasta",
     "KRAS (UniProt P01116)"),
    (SEQUENCES / "P01112.fasta",
     "HRAS (UniProt P01112)"),
    (SEQUENCES / "P01111.fasta",
     "NRAS (UniProt P01111)"),

    # Estructuras PDB
    (PDB / "pdb5uk9.ent",
     "KRAS·GCP (PDB 5UK9, ligando estructura análoga de GTP, resolución 1.89 Å)"),
    (PDB / "pdb4obe.ent",
     "KRAS·GDP (PDB 4OBE, ligando GDP, resolución 1.24 Å)"),
    (PDB / "pdb3K8Y.ent",
     "HRAS·GNP (PDB 3K8Y, ligando estructura análoga de GTP, resolución 1.30 Å)"),
    (PDB / "pdb4q21.ent",
     "HRAS·GDP (PDB 4Q21, ligando GDP, resolución 2.00 Å)"),
    (PDB / "pdb5uhv.ent",
     "NRAS·GNP (PDB 5UHV, ligando estructura análoga de GTP, resolución 1.67 Å)"),
    (PDB / "pdb3con.ent",
     "HRAS·GDP (PDB 3CON, ligando GDP, resolución 1.65 Å)"),
]

all_files_ok = True  # bandera global de ficheros

for file_path, description in REQUIRED_FILES:

    if file_path.exists():  # .exists() devuelve True si el fichero está en disco
        # .stat().st_size devuelve el tamaño del fichero en bytes.
        size_kb = file_path.stat().st_size / 1024  # convertimos a kilobytes
        
        try:
            display_path = file_path.relative_to(ROOT)
        except ValueError:
            display_path = file_path

        print(f"  [OK] {str(display_path):<75} {size_kb:>10.1f} KB")

    else:

        try:
            display_path = file_path.relative_to(ROOT)
        except ValueError:
            display_path = file_path

        print(f"  [FALTA] {display_path}")
        print(f"           → {description}")

        all_files_ok = False

print("-" * 60)

if all_files_ok:
    print("Todos los ficheros de entrada están presentes.")
else:
    print("ADVERTENCIA: faltan uno o más ficheros de entrada.")
    print("Ejecuta 'python scripts/run_all.py --use-synthetic' o verifica la carpeta de datos.")


Verificando ficheros de datos de entrada...
------------------------------------------------------------
  [OK] data/raw/cosmic/Cosmic_MutantCensus_Tsv_v104_GRCh38/Cosmic_MutantCensus_v104_GRCh38.tsv.gz   115320.2 KB
  [OK] data/raw/cosmic/Cosmic_Classification_Tsv_v104_GRCh38/Cosmic_Classification_v104_GRCh38.tsv.gz      154.3 KB
  [OK] data/external/sequences/P01116.fasta                                               0.3 KB
  [OK] data/external/sequences/P01112.fasta                                               0.3 KB
  [OK] data/external/sequences/P01111.fasta                                               0.3 KB
  [OK] data/external/pdb/pdb5uk9.ent                                                    271.2 KB
  [OK] data/external/pdb/pdb4obe.ent                                                    732.9 KB
  [OK] data/external/pdb/pdb3K8Y.ent                                                    166.9 KB
  [OK] data/external/pdb/pdb4q21.ent                                                 

---
## Sección 6: Herramientas externas

El pipeline puede usar herramientas externas para ciertas operaciones, pero **tiene alternativas integradas** si no están disponibles:

| Herramienta | Uso | Alternativa integrada |
|-------------|-----|----------------------|
| `clustalo` (Clustal Omega) | MSA de secuencias | Alineamiento directo (sin gaps) |
| `mafft` | MSA alternativo | Idem |
| `mkdssp` / `dssp` | Cálculo de SASA y estructura secundaria | Shrake-Rupley (Biopython) |


In [98]:
# ============================================================
# CHUNK 6.1: Verificar herramientas externas
# ============================================================

import shutil  # módulo que permite buscar ejecutables en el PATH del sistema

print("Verificando herramientas externas...")
print("-" * 50)

# shutil.which(nombre) busca el ejecutable en el PATH del sistema.
# Devuelve la ruta completa si lo encuentra, o None si no está instalado.
# Es equivalente a ejecutar 'which clustalo' en la terminal.

TOOLS = [
    # (nombre_ejecutable, descripción, ¿es_obligatorio?)
    ("clustalo",  "Clustal Omega — MSA de proteínas",  False),
    ("mafft",     "MAFFT — MSA alternativo",            False),
    ("muscle",    "MUSCLE — MSA alternativo",           False),
    ("mkdssp",    "DSSP — SASA y estructura secundaria", False),
    ("dssp",      "DSSP (nombre alternativo)",           False),
]

msa_tool_found = False   # bandera: ¿encontramos al menos una herramienta MSA?
dssp_found = False       # bandera: ¿encontramos DSSP?

for tool_name, description, required in TOOLS:
    path = shutil.which(tool_name)  # busca el ejecutable en el PATH

    if path:  # si path no es None, el ejecutable existe
        print(f"  [OK]    {tool_name:12s} → {path}")
        if tool_name in ("clustalo", "mafft", "muscle"):
            msa_tool_found = True  # marcamos que hay una herramienta MSA
        if tool_name in ("mkdssp", "dssp"):
            dssp_found = True  # marcamos que DSSP está disponible
    else:
        print(f"  [N/D]   {tool_name:12s} — no instalado (se usará alternativa integrada)")

print("-" * 50)
print()

# Resumen del estado de las herramientas externas.
if msa_tool_found:
    print("  MSA: herramienta externa disponible.")
else:
    print("  MSA: usando alineamiento directo (sin software externo).")
    print("       Esto es correcto para KRAS/HRAS/NRAS (secuencias >88% idénticas).")

if dssp_found:
    print("  SASA: DSSP disponible (método más preciso).")
else:
    print("  SASA: usando Shrake-Rupley (implementación nativa de BioPython).")
    print("       Los resultados son equivalentes para este TFM.")

Verificando herramientas externas...
--------------------------------------------------
  [OK]    clustalo     → /Users/rachi/miniforge3/envs/tfm-ras/bin/clustalo
  [OK]    mafft        → /Users/rachi/miniforge3/envs/tfm-ras/bin/mafft
  [N/D]   muscle       — no instalado (se usará alternativa integrada)
  [OK]    mkdssp       → /Users/rachi/miniforge3/envs/tfm-ras/bin/mkdssp
  [N/D]   dssp         — no instalado (se usará alternativa integrada)
--------------------------------------------------

  MSA: herramienta externa disponible.
  SASA: DSSP disponible (método más preciso).


---
## Sección 7: Directorios de salida

Los notebooks de análisis generan ficheros en dos directorios:

- `data/processed/`: ficheros CSV con los resultados del pipeline (datasets curados, mapa posicional, tabla maestra, tablas de resultados).
- `figures/`: figuras PNG, SVG y HTML generadas por los notebooks 02, 03 y 04.

Esta sección verifica que ambos directorios existen y los crea si no están presentes.

In [99]:
# ============================================================
# CHUNK 7.1: Crear directorios de salida si no existen
# ============================================================

print("Verificando directorios de salida...")
print("-" * 40)

# Definimos los directorios de salida que necesita el pipeline.
# Son rutas relativas a la raíz del proyecto (ROOT).
OUTPUT_DIRS = [
    ROOT / "data" / "processed",   # CSVs procesados del pipeline
    ROOT / "figures",               # figuras PNG, SVG, HTML
]

for dir_path in OUTPUT_DIRS:
    if dir_path.exists():
        # .is_dir() verifica que la ruta existe Y es un directorio (no un fichero).
        if dir_path.is_dir():
            print(f"  [OK] {dir_path.relative_to(ROOT)}")
        else:
            print(f"  [ERROR] {dir_path.relative_to(ROOT)} existe pero no es un directorio.")
    else:
        # .mkdir(parents=True, exist_ok=True) crea el directorio y los intermedios
        # si no existen; exist_ok=True evita error si ya existe (condición de carrera).
        dir_path.mkdir(parents=True, exist_ok=True)  # creamos el directorio
        print(f"  [CREADO] {dir_path.relative_to(ROOT)}")

print("-" * 40)
print("Directorios de salida listos.")

Verificando directorios de salida...
----------------------------------------
  [OK] data/processed
  [OK] figures
----------------------------------------
Directorios de salida listos.


---
## Sección 8: Resumen del entorno

Esta sección consolida todos los resultados de las verificaciones anteriores y proporciona una recomendación clara sobre si el entorno está listo para ejecutar el pipeline.

In [100]:
# ============================================================
# CHUNK 8.1: Tabla del entorno computacional para metodología
# ============================================================
# Generamos una tabla con los recursos bioinformáticos utilizados
# en el pipeline, sus versiones y su función en el análisis.
# Se guarda como figura PNG en figures/ para incluir en el TFM.
import importlib
import sys
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# Obtener versiones de librerías Python automáticamente
def _get_version(package):
    try:
        return importlib.metadata.version(package)
    except Exception:
        try:
            mod = importlib.import_module(package)
            return getattr(mod, "__version__", "N/D")
        except Exception:
            return "N/D"

python_version = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"

# Definición de la tabla de recursos
resources = [
    # Lenguaje y entorno
    ["Python",          python_version,              "Lenguaje",   "Lenguaje base del pipeline"],
    ["conda-forge",     "26.1.0",                    "Entorno",    "Gestión del entorno y dependencias"],

    # Librerías de análisis de datos
    ["pandas",          _get_version("pandas"),       "Librería",   "Manipulación y curación del dataset COSMIC"],
    ["NumPy",           _get_version("numpy"),        "Librería",   "Cálculo vectorizado de distancias (broadcasting)"],

    # Bioinformática estructural
    ["BioPython",       _get_version("biopython"),    "Librería",   "Parseo PDB, MSA, SASA (Shrake-Rupley), SeqIO"],

    # Visualización
    ["Matplotlib",      _get_version("matplotlib"),   "Librería",   "Generación de figuras y tabla del MSA"],
    ["seaborn",         _get_version("seaborn"),      "Librería",   "Heatmaps y visualizaciones estadísticas"],
    ["py3Dmol",         _get_version("py3Dmol"),      "Librería",   "Visualización molecular 3D interactiva"],

    # Bases de datos externas
    ["COSMIC v.104",    "v.104",                      "Base datos", "Mutaciones somáticas missense de KRAS/HRAS/NRAS"],
    ["RCSB PDB",        "REST API (2026)",            "Base datos", "Estructuras cristalográficas 3D (4OBE, 3K8Y, 5UHV)"],
    ["UniProt",         "REST API (2026)",            "Base datos", "Secuencias canónicas y anotaciones funcionales"],
]

# Columnas de la tabla
columns = ["Recurso", "Versión", "Tipo", "Uso en el pipeline"]

# Guardar como CSV
import pandas as pd
df_resources = pd.DataFrame(resources, columns=columns)

ROOT = project_root()
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

df_resources.to_csv(FIGURES / "computational_environment.csv", index=False)
print("[OK] Tabla guardada en figures/computational_environment.csv")

# --- Generar figura PNG ------------------------------------------
# Colores por tipo de recurso para distinguir visualmente las categorías.
TYPE_COLORS = {
    "Lenguaje":   "#4C78A8",   # azul
    "Entorno":    "#72B7B2",   # verde azulado
    "Librería":   "#54A24B",   # verde
    "Base datos": "#F58518",   # naranja
}

# Color de fondo de cada fila según el tipo de recurso.
row_colors = [
    [TYPE_COLORS.get(row[2], "#DDDDDD")] + ["#F9F9F9"] * (len(columns) - 1)
    for row in resources
]

fig, ax = plt.subplots(figsize=(14, len(resources) * 0.55 + 1.2))
ax.axis("off")

# Construimos la tabla con matplotlib.table
table = ax.table(
    cellText=resources,
    colLabels=columns,
    cellLoc="left",
    loc="center",
    cellColours=row_colors,
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.auto_set_column_width(col=list(range(len(columns))))

# Estilo de cabecera
for col_idx in range(len(columns)):
    cell = table[0, col_idx]
    cell.set_facecolor("#2C3E50")    # gris oscuro
    cell.set_text_props(color="white", fontweight="bold")
    cell.set_height(0.08)

# Altura uniforme para filas de datos
for row_idx in range(1, len(resources) + 1):
    for col_idx in range(len(columns)):
        table[row_idx, col_idx].set_height(0.06)

# Título y leyenda de tipos
ax.set_title(
    "Entorno computacional del pipeline bioinformático",
    fontsize=11, fontweight="bold", pad=12, loc="left"
)

legend_handles = [
    mpatches.Patch(color=color, label=tipo)
    for tipo, color in TYPE_COLORS.items()
]
ax.legend(
    handles=legend_handles,
    loc="lower right",
    fontsize=8,
    title="Tipo de recurso",
    title_fontsize=8,
    framealpha=0.9,
    edgecolor="#cccccc",
)

fig.tight_layout()
fig.savefig(FIGURES / "computational_environment.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print("[OK] Figura guardada en figures/computational_environment.png")
print()
print(df_resources.to_string(index=False))

[OK] Tabla guardada en figures/computational_environment.csv


[OK] Figura guardada en figures/computational_environment.png

     Recurso         Versión       Tipo                                 Uso en el pipeline
      Python         3.11.15   Lenguaje                         Lenguaje base del pipeline
 conda-forge          26.1.0    Entorno                 Gestión del entorno y dependencias
      pandas           3.0.3   Librería         Manipulación y curación del dataset COSMIC
       NumPy           2.4.6   Librería   Cálculo vectorizado de distancias (broadcasting)
   BioPython            1.87   Librería       Parseo PDB, MSA, SASA (Shrake-Rupley), SeqIO
  Matplotlib          3.10.9   Librería              Generación de figuras y tabla del MSA
     seaborn          0.13.2   Librería            Heatmaps y visualizaciones estadísticas
     py3Dmol           2.5.4   Librería             Visualización molecular 3D interactiva
COSMIC v.104           v.104 Base datos    Mutaciones somáticas missense de KRAS/HRAS/NRAS
    RCSB PDB REST API (2026

In [101]:
# ============================================================
# CHUNK 8.2: Resumen final del estado del entorno
# ============================================================

print("=" * 55)
print("RESUMEN DEL ENTORNO")
print("=" * 55)
print()

# Importamos directamente los módulos principales para confirmar
# que todo el stack funciona de extremo a extremo.
try:
    from tfm_ras import cosmic_loader, alignment, structural, visualization
    from tfm_ras.config import project_root, load_config
    import pandas as pd
    import numpy as np
    from Bio import SeqIO  # módulo de BioPython para parseo de FASTA
    import matplotlib

    print("  [OK] Importaciones principales: PASS")
    
    print()
    print("=" * 55)
    print("ENTORNO LISTO")
    print("=" * 55)
    print()
    print("Puedes proceder con los notebooks de análisis:")
    print()
    print("  Hito 1 → notebooks/01_data_curation.ipynb")
    print("  Hito 2 → notebooks/02_alignment_mapping.ipynb")
    print("  Hito 3 → notebooks/03_structural_features.ipynb")
    print("  Hito 4 → notebooks/04_results_visualization.ipynb")
    print()
    print("Alternativamente, ejecuta todo el pipeline de una vez:")
    print("  python scripts/run_all.py --use-synthetic")

except Exception as e:
    # Capturamos cualquier excepción para dar un mensaje de error claro.
    print(f"  [ERROR] {type(e).__name__}: {e}")
    print()
    print("El entorno NO está listo. Revisa los errores anteriores y:")
    print("  1. conda activate tfm-ras")
    print("  2. pip install -e .")
    print("  3. Vuelve a ejecutar este notebook desde el principio")

RESUMEN DEL ENTORNO

  [OK] Importaciones principales: PASS

ENTORNO LISTO

Puedes proceder con los notebooks de análisis:

  Hito 1 → notebooks/01_data_curation.ipynb
  Hito 2 → notebooks/02_alignment_mapping.ipynb
  Hito 3 → notebooks/03_structural_features.ipynb
  Hito 4 → notebooks/04_results_visualization.ipynb

Alternativamente, ejecuta todo el pipeline de una vez:
  python scripts/run_all.py --use-synthetic


---
## Referencia: estructura del proyecto

```
plantilla_v2/
├── configs/
│   └── config.yaml              ← parámetros del pipeline
├── data/
│   ├── raw/cosmic/
│   │   └── synthetic_cosmic.tsv ← dataset sintético (incluido en el repo)
│   ├── external/
│   │   ├── sequences/           ← FASTA canónicos de UniProt
│   │   └── pdb/                 ← estructuras PDB descargadas
│   └── processed/               ← CSVs generados por el pipeline
├── figures/                     ← figuras PNG, SVG, HTML
├── notebooks/
│   ├── 00_setup.ipynb           ← este notebook (diagnóstico del entorno)
│   ├── 01_data_curation.ipynb   ← Hito 1: curación COSMIC
│   ├── 02_alignment_mapping.ipynb ← Hito 2: MSA y mapa posicional
│   ├── 03_structural_features.ipynb ← Hito 3: SASA, distancia, entropía
│   └── 04_results_visualization.ipynb ← Hito 4: figuras y tablas finales
├── src/tfm_ras/                 ← código fuente del paquete
│   ├── __init__.py
│   ├── config.py
│   ├── cosmic_loader.py
│   ├── alignment.py
│   ├── structural.py
│   └── visualization.py
├── tests/                       ← suite de tests automatizados
├── scripts/
│   └── run_all.py               ← ejecuta todos los notebooks en secuencia
└── docs/                        ← documentación del TFM
```
